In [3]:
# Article Bias Analyzer v3 — Professional Interactive Notebook

**Purpose**
This notebook is the cleaned, tool-style backend for the project. It takes one article and returns a structured journalism analysis report.

**Input**
- Either paste article text into `article_text`
- Or paste a link into `article_url`

**Output**
- source type and likely outlet
- likely people / subjects in the article
- likely speakers and what they said
- sentence-by-sentence sentiment, bias, context, and quote analysis
- emotion intensity %
- analysis confidence %
- article-level summary
- 3_4 line quick professional summary

**Important note**
- This notebook reports **confidence estimates**, not true accuracy.
- True accuracy for a new unseen article cannot be known without human-labeled ground truth.
- The notebook is designed to be stable, readable, and interpretable for the current stage of the project.

**Current strengths**
- sentiment + bias separation
- implicit / explicit cue detection
- some context-aware adjustment
- quote vs narration distinction
- clean report-style output

**Current limits**
- subtle sarcasm can still be missed
- deeper multi-sentence framing is only partly captured
- source scraping depends on public webpage structure"

SyntaxError: unterminated string literal (detected at line 35) (2168678587.py, line 35)

In [4]:
!pip install -q textblob vaderSentiment pandas requests beautifulsoup4

In [5]:
import re
import math
import urllib.parse
import pandas as pd
import requests
from bs4 import BeautifulSoup
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

# =========================
# Core dictionaries / rules
# =========================

IMPLICIT_NEGATIVE_CUES = [
    "declined to comment",
    "no plan was offered",
    "was not addressed",
    "remains unclear",
    "failed to",
    "did not explain",
    "has yet to",
    "warned that",
    "concerns remain",
    "under pressure",
    "critics say",
    "questions remain",
    "without explanation",
    "did not respond",
    "doubts about",
    "must be revealed",
    "can only be revealed",
    "accused",
    "criticized",
    "wiped out opposition",
    "in exile or dead",
    "must resign",
    "not a legitimate president"
]

EXPLICIT_NEGATIVE_CUES = [
    "disgrace",
    "failure",
    "shameful",
    "hypocritical",
    "catastrophe",
    "collapse",
    "threat",
    "crisis",
    "war criminal",
    "thief",
    "dead-end war",
    "enormous damage",
    "lost his mind",
    "insane",
    "morbid craving"
]

POSITIVE_CUES = [
    "welcomed",
    "successful",
    "praised",
    "progress",
    "improved",
    "encouraging",
    "beneficial",
    "victory",
    "support"
]

SARCASM_MARKERS = [
    "oh great",
    "yeah,",
    "yeah ",
    "sure,",
    "sure ",
    "as if",
    "exactly what we needed",
    "i'm impressed",
    "i am impressed",
    "brilliant decision",
    "fantastic job",
    "amazing leadership",
    "what a great"
]

NEG_CONTEXT_WORDS = [
    "delay",
    "failure",
    "problem",
    "crisis",
    "war",
    "damage",
    "lost",
    "dead",
    "hostage",
    "criticized",
    "accused",
    "repression",
    "concern",
    "worse"
]

STOP_PHRASES = {
    "United States",
    "New York",
    "White House",
    "Google Colab",
    "Media Bias",
    "Bias Annotation",
    "TextBlob",
    "NBC News"
}

SOURCE_MAP = {
    "reuters.com": ("news_website", "Reuters"),
    "apnews.com": ("news_website", "Associated Press"),
    "bbc.com": ("news_website", "BBC"),
    "cnn.com": ("news_website", "CNN"),
    "nbcnews.com": ("news_website", "NBC News"),
    "nytimes.com": ("news_website", "New York Times"),
    "theguardian.com": ("news_website", "The Guardian"),
    "washingtonpost.com": ("news_website", "The Washington Post"),
    "foxnews.com": ("news_website", "Fox News"),
    "x.com": ("social_media", "X / Twitter"),
    "twitter.com": ("social_media", "X / Twitter"),
    "facebook.com": ("social_media", "Facebook"),
    "reddit.com": ("social_media", "Reddit"),
    "instagram.com": ("social_media", "Instagram"),
    "youtube.com": ("social_media", "YouTube"),
    "tiktok.com": ("social_media", "TikTok")
}

SPEAKER_PATTERNS = [
    re.compile(r'\b([A-Z][A-Za-z.\-]+(?:\s+[A-Z][A-Za-z.\-]+){0,2})\s+(said|wrote|added|told|argued|warned|criticized|accused|responded|called|urged|noted|discussed)\b'),
    re.compile(r'\b(?:according to|said|wrote|added|told|argued|warned|criticized|accused|responded|called|urged|noted)\s+([A-Z][A-Za-z.\-]+(?:\s+[A-Z][A-Za-z.\-]+){0,2})\b')
]

# =========================
# Helper functions
# =========================

def classify_source(domain: str):
    domain = domain.lower().replace("www.", "")
    if not domain:
        return "pasted_text", "Pasted Article Text"
    for key, value in SOURCE_MAP.items():
        if domain.endswith(key):
            return value
    return "website", domain

def get_article_text_and_meta(article_text: str, article_url: str):
    meta = {
        "title": "",
        "domain": "",
        "source_type": "pasted_text",
        "source_name": "Pasted Article Text",
        "scrape_status": "pasted_text"
    }

    if isinstance(article_text, str) and article_text.strip():
        return article_text.strip(), meta

    if isinstance(article_url, str) and article_url.strip():
        parsed = urllib.parse.urlparse(article_url.strip())
        domain = parsed.netloc
        source_type, source_name = classify_source(domain)
        meta["domain"] = domain
        meta["source_type"] = source_type
        meta["source_name"] = source_name

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        try:
            response = requests.get(article_url.strip(), headers=headers, timeout=20)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            title_tag = soup.find("title")
            if title_tag:
                meta["title"] = title_tag.get_text(" ", strip=True)

            # Try article tags first
            article_blocks = soup.find_all(["article"])
            paragraphs = []

            if article_blocks:
                for block in article_blocks:
                    paragraphs.extend([p.get_text(" ", strip=True) for p in block.find_all("p")])

            # fallback to all paragraph tags
            if not paragraphs:
                paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]

            cleaned = []
            for p in paragraphs:
                p = re.sub(r"\s+", " ", p).strip()
                if len(p) >= 40:
                    cleaned.append(p)

            article = "\n".join(cleaned).strip()

            if not article:
                meta["scrape_status"] = "no_text_extracted"
            else:
                meta["scrape_status"] = "success"

            return article, meta

        except Exception as e:
            meta["scrape_status"] = f"scrape_error: {e}"
            return "", meta

    return "", meta

def split_sentences_clean(text: str):
    text = text.replace("\r", "\n").replace("“", '"').replace("”", '"').replace("’", "'").strip()

    # protect common abbreviations and initials
    protected = {
        "U.S.": "US_PROTECT",
        "U.K.": "UK_PROTECT",
        "J.F.K.": "JFK_PROTECT",
        "Mr.": "MR_PROTECT",
        "Mrs.": "MRS_PROTECT",
        "Dr.": "DR_PROTECT",
        "Prof.": "PROF_PROTECT",
        "No.": "NO_PROTECT",
        "St.": "ST_PROTECT"
    }

    for k, v in protected.items():
        text = text.replace(k, v)

    # protect single initials like "J."
    text = re.sub(r"\b([A-Z])\.", r"\1_INITIALPROTECT", text)

    text = re.sub(r"\s+", " ", text)

    chunks = re.split(r'[.!?]+["\']?\s+(?=[A-Z])', text)
    chunks = [c.strip() for c in chunks if c.strip()]

    restored = []
    for c in chunks:
        for k, v in protected.items():
            c = c.replace(v, k)
        c = c.replace("_INITIALPROTECT", ".")
        restored.append(c)

    merged = []
    i = 0
    while i < len(restored):
        cur = restored[i]

        if i + 1 < len(restored):
            tail_word = cur.split()[-1].lower() if cur.split() else ""
            if len(cur.split()) <= 3 or tail_word in {
                "the", "a", "an", "of", "to", "on", "in", "with", "and", "including", "saying"
            }:
                merged.append((cur + " " + restored[i + 1]).strip())
                i += 2
                continue

        merged.append(cur)
        i += 1

    final = []
    for s in merged:
        s = s.strip()
        if not s:
            continue
        if final and len(s.split()) <= 1:
            final[-1] += " " + s
        else:
            final.append(s)

    return final

def detect_quote_type(text: str):
    if any(q in text for q in ['"', "“", "”", "‘", "’"]):
        return "quote_or_quoted_content"
    return "narration"

def extract_speaker(text: str):
    for pattern in SPEAKER_PATTERNS:
        match = pattern.search(text)
        if match:
            return match.group(1)
    return ""

def extract_subjects(full_text: str, topn: int = 5):
    matches = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}\b', full_text)

    bad_single = {
        "On", "In", "When", "The", "A", "An", "He", "She", "It",
        "Meanwhile", "But", "Among", "Even", "Tuesday", "Thursday",
        "Friday", "Wednesday", "Monday", "Saturday", "Sunday"
    }

    counts = {}
    for match in matches:
        parts = match.split()

        if match in STOP_SUBJECTS:
            continue
        if parts[0] in bad_single:
            continue
        if len(parts) == 1 and match not in {
            "Russia", "Ukraine", "Kremlin", "Putin", "Remeslo", "Navalny",
            "Telegram", "Fontanka", "Tass", "Solovyov", "Rubio", "Trump",
            "Iran", "Israel", "Hezbollah", "Microsoft"
        }:
            continue

        counts[match] = counts.get(match, 0) + 1

    return sorted(counts, key=counts.get, reverse=True)[:topn]

def sentiment_label(tb_polarity: float, vd_compound: float):
    if vd_compound <= -0.1 or tb_polarity <= -0.1:
        return "negative"
    if vd_compound >= 0.1 or tb_polarity >= 0.1:
        return "positive"
    return "neutral"

def analyze_sentence(text: str, prev_text: str = "", next_text: str = ""):
    tb = TextBlob(text).sentiment
    vd = vader.polarity_scores(text)

    context_window = " ".join([prev_text, text, next_text]).strip()
    wtb = TextBlob(context_window).sentiment
    wvd = vader.polarity_scores(context_window)

    text_lower = text.lower()

    implicit_hits = [p for p in IMPLICIT_NEGATIVE_CUES if p in text_lower]
    explicit_hits = [p for p in EXPLICIT_NEGATIVE_CUES if p in text_lower]
    positive_hits = [p for p in POSITIVE_CUES if p in text_lower]
    sarcasm_hits = [p for p in SARCASM_MARKERS if p in text_lower]

    sarcasm_flag = False
    if sarcasm_hits:
        sarcasm_flag = True
    if tb.polarity > 0.2 and vd["compound"] < -0.3:
        sarcasm_flag = True

    sent_label = sentiment_label(tb.polarity, vd["compound"])

    # strong override: if explicit or implicit hits exist, sentiment should not remain positive
    if explicit_hits or implicit_hits:
        sent_label = "negative"

    context_adjusted_label = sent_label
    if sent_label == "neutral" and (wvd["compound"] <= -0.15 or wtb.polarity <= -0.1 or implicit_hits or explicit_hits):
        context_adjusted_label = "context_negative"

    bias_label = "none"
    if explicit_hits:
        bias_label = "explicit_negative"
    elif implicit_hits:
        bias_label = "implicit_negative"
    elif context_adjusted_label == "context_negative":
        bias_label = "contextual_negative"

    if sarcasm_flag:
        bias_label = "sarcasm" if bias_label == "none" else f"{bias_label}+sarcasm"

    effective_for_emotion = context_adjusted_label if context_adjusted_label != "context_negative" else "negative"
    emotion_label = estimate_emotion_label(text, effective_for_emotion)

    emotion_intensity = round(
        min(100, max(abs(wvd["compound"]), abs(wtb.polarity)) * 70 + max(tb.subjectivity, wtb.subjectivity) * 30),
        1
    )

    confidence = 50
    confidence += abs(vd["compound"]) * 15
    confidence += abs(wvd["compound"]) * 10
    confidence += abs(tb.polarity) * 10
    confidence += len(implicit_hits) * 5
    confidence += len(explicit_hits) * 8
    confidence += 7 if sarcasm_flag else 0

    # downgrade weak confidence if only one weak signal and neutral base
    if sent_label == "neutral" and not explicit_hits and len(implicit_hits) <= 1 and abs(vd["compound"]) < 0.2:
        confidence -= 8

    confidence = round(max(45, min(98, confidence)), 1)

    matched = []
    matched += [f"implicit:{x}" for x in implicit_hits]
    matched += [f"explicit:{x}" for x in explicit_hits]
    matched += [f"positive:{x}" for x in positive_hits]
    matched += [f"sarcasm:{x}" for x in sarcasm_hits]

    return {
        "sentence": text,
        "speaker": extract_speaker(text),
        "subject_focus": ", ".join(extract_subjects(text, 2)),
        "quote_type": detect_quote_type(text),
        "textblob_polarity": round(tb.polarity, 3),
        "textblob_subjectivity": round(tb.subjectivity, 3),
        "vader_compound": round(vd["compound"], 3),
        "sentiment_label": sent_label,
        "context_adjusted_label": context_adjusted_label,
        "bias_label": bias_label,
        "emotion_label": emotion_label,
        "emotion_intensity_%": emotion_intensity,
        "analysis_confidence_%": confidence,
        "matched_cues": "; ".join(matched)
    }


def summarize_article(df: pd.DataFrame, source_name: str, source_type: str, title: str, top_subjects):
    total = len(df)
    count_positive = int((df["sentiment_label"] == "positive").sum())
    count_negative = int((df["sentiment_label"] == "negative").sum())
    count_neutral = int((df["sentiment_label"] == "neutral").sum())
    count_context_negative = int((df["context_adjusted_label"] == "context_negative").sum())
    count_bias_flagged = int((df["bias_label"] != "none").sum())
    count_quotes = int((df["quote_type"] == "quote_or_quoted_content").sum())

    avg_emotion = round(float(df["emotion_intensity_%"].mean()), 1) if total else 0.0
    avg_conf = round(float(df["analysis_confidence_%"].mean()), 1) if total else 0.0

    if count_bias_flagged >= max(2, math.ceil(total * 0.20)):
        overall = "neutral tone with noticeable bias/framing signals"
    elif count_negative > max(count_positive, count_neutral):
        overall = "mostly negative"
    elif count_positive > max(count_negative, count_neutral):
        overall = "mostly positive"
    else:
        overall = "mostly neutral"

    summary_lines = []
    summary_lines.append(f"Source: {source_name} ({source_type}).")
    if title:
        summary_lines.append(f"Headline/title detected: {title}.")
    if top_subjects:
        summary_lines.append(f"Main subject focus: {', '.join(top_subjects[:3])}.")
    summary_lines.append(
        f"The article appears {overall}, with {count_bias_flagged} bias-flagged sentences, {count_context_negative} context-adjusted negative sentences, and average emotion intensity of {avg_emotion}%."
    )

    return {
        "source_name": source_name,
        "source_type": source_type,
        "title": title,
        "top_subjects": ", ".join(top_subjects),
        "total_sentences": total,
        "positive_sentences": count_positive,
        "negative_sentences": count_negative,
        "neutral_sentences": count_neutral,
        "context_negative_sentences": count_context_negative,
        "bias_flagged_sentences": count_bias_flagged,
        "quote_or_quoted_sentences": count_quotes,
        "average_emotion_intensity_%": avg_emotion,
        "average_analysis_confidence_%": avg_conf,
        "overall_article_interpretation": overall,
        "quick_summary": " ".join(summary_lines)
    }

print("Model loaded successfully.")

Model loaded successfully.


EXAMPLE ARTICLE TEXT

In [6]:
# Fill ONE of these only. Leave the other empty.

article_text = """A loyal pro-Kremlin blogger who unexpectedly denounced Russian President Vladimir Putin as 'a war criminal and thief'
has been admitted to a psychiatric facility. Ilya Remeslo, a lawyer and pro-Putin firebrand, shocked both Kremlin supporters and critics
Tuesday when he shared a long post on Telegram, titled 'Five reasons why I stopped supporting Vladimir Putin.'
Among his reasons, he outlined Russia’s invasion of Ukraine, which he called an 'absolutely dead-end war.'
He also blamed the Kremlin for 'enormous damage to the Russian economy and the well-being of citizens' and criticized the Russian government’s campaign to throttle internet and digital freedoms, including
the anticipated ban of Telegram, the country’s most popular messenger app.
Remeslo, previously known as a vocal critic of the late opposition leader Alexei Navalny who even testified against him in court, accused Putin of being in power for too long, with apparent plans 'to sit on the
throne for at least 150 years.'
Even a 'morally impeccable person' would be corrupted by such a long reign, he said.
Putin does not respect his voters and does not want to listen to them, Remeslo wrote, adding that the Russian leader has wiped out opposition, and that anyone who has dared to speak out is either in exile or dead.
'Bottom line. Vladimir Putin is not a legitimate president. Vladimir Putin must resign and be brought to trial as a war criminal and thief,” he wrote.
Remeslo’s comments were an unusual public display of dissent and personal criticism of Putin in an atmosphere of repression and tight control in Russia, exacerbated by the invasion of Ukraine.
So abrupt was the change that some in the pro-Kremlin camp speculated that Remeslov’s account might have been hacked, that he was being held hostage or that he had for some other reason just lost the plot.
Remeslo responded in posts on Telegram that he had not been hacked, that he remained in Russia and stood by his opinions.
On Wednesday, he continued his criticism, accusing Putin of an 'insane, borderline morbid craving for luxury' as he referenced anti-corruption investigations into the leader's assets, of the kind pioneered by Navalny.
He also recorded a video saying that Putin was too afraid to surround himself with those who can tell him the honest truth.
In a response to the Russian media outlet Ostoroghno Novosti, Remeslo said his views changed because the country has 'changed a lot.'
Prominent pro-Kremlin TV host Vladimir Solovyov discussed Remeslo on his show Wednesday, referring to him as 'a lawyer who has lost his mind,' without naming him, saying “some people’s nerves can’t take it.”
Responding to Solovyov’s criticism, Remeslo urged him to switch to the 'side of light.'
Meanwhile, some opponents of the Kremlin expressed doubts about Remeslo’s motives, and whether he had already fallen out of favor with the Kremlin.
But the story took a dramatic turn after Remeslo abruptly stopped posting late Wednesday. On Thursday, the news website Fontanka reported that Remeslo has been hospitalized at Skvortsov-Stepanov Psychiatric Hospital No. 3 in St. Petersburg.
When NBC News called the hospital Friday, a man who picked up the phone but did not want to share his name said that someone matching the name of Ilya Remeslo was indeed a patient at the hospital.
He was admitted Thursday, but the grounds for his hospitalization can only be revealed to the patient’s family, NBC News was told.
Remeslo did not respond to a request for comment. The blogger, 42, was previously a vocal critic of Navalny, who died in an Arctic penal colony in early 2024.
The state news agency Tass called Remeslo 'one of Navalny’s most famous whistleblowers' as he testified against the politician in court in 2022 and investigated his anti-corruption fund, FBK.
In one of the posts Wednesday, however, he signed off with a tagline made famous by the late opposition leader, concluding: 'Here we tell the truth.
"""

article_url = ""

In [7]:
# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
# propagate speaker forward where the next sentence likely belongs to same speaking flow
df["speaker"] = df["speaker"].replace("", pd.NA).ffill().fillna("")

# framing trend across nearby sentences
def framing_score(row):
    if row["context_adjusted_label"] in ["negative", "context_negative"]:
        return -1
    elif row["sentiment_label"] == "positive":
        return 1
    return 0

df["framing_score"] = df.apply(framing_score, axis=1)
df["rolling_framing"] = df["framing_score"].rolling(3, min_periods=1).mean()

# combined severity score for ranking
def severity_score(row):
    score = 0
    if row["bias_label"] == "explicit_negative":
        score += 4
    elif row["bias_label"] == "implicit_negative":
        score += 3
    elif row["bias_label"] == "contextual_negative":
        score += 2
    elif "sarcasm" in str(row["bias_label"]):
        score += 3

    if row["sentiment_label"] == "negative":
        score += 2
    if row["context_adjusted_label"] == "context_negative":
        score += 1

    score += row["analysis_confidence_%"] / 100
    score += row["emotion_intensity_%"] / 200
    return round(score, 3)

df["severity_score"] = df.apply(severity_score, axis=1)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = df[df["bias_label"] != "none"].copy()
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values(["severity_score", "vader_compound"], ascending=[False, True]).head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()

print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

NameError: name 'estimate_emotion_label' is not defined

CURRENT NEWS ARTICLE 1

In [ ]:
# Fill ONE of these only. Leave the other empty.

article_text = """
"""

article_url = "https://www.cnn.com/2026/04/07/middleeast/pro-iran-militia-in-iraq-releases-american-journalist-latam-intl"

# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = df[df["bias_label"] != "none"].copy()
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values("vader_compound").head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()

print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

AI JOURNALISM ANALYSIS REPORT
Source Type: news_website
Likely Outlet / Platform: CNN
Detected Title: Pro-Iran militia in Iraq says they have decided to release American journalist | CNN
Main Subject(s): Kataib Hezbollah, Shelly Kittleson, Middle East, Cable News Network, State Marco Rubio
Total Sentences: 27
Overall Interpretation: mostly positive
Average Emotion Intensity: 29.7%
Average Analysis Confidence: 58.4%

QUICK SUMMARY
--------------------------------------------------------------------------------
Source: CNN (news_website). Headline/title detected: Pro-Iran militia in Iraq says they have decided to release American journalist | CNN. Main subject focus: Kataib Hezbollah, Shelly Kittleson, Middle East. The article appears mostly positive, with 1 bias-flagged sentences, 1 context-adjusted negative sentences, and average emotion intensity of 29.7%.

SENTIMENT COUNTS
--------------------------------------------------------------------------------
sentiment_label
positive    13


,sentence,speaker,subject_focus,quote_type,textblob_polarity,textblob_subjectivity,vader_compound,sentiment_label,context_adjusted_label,bias_label,emotion_intensity_%,analysis_confidence_%,matched_cues
0,US Secretary of State Marco Rubio said Tuesday...,State Marco Rubio,"State Marco Rubio, Shelly Kittleson",narration,0.000,0.000,0.000,neutral,neutral,none,0.0,50.0,
1,Kittleson was received by the Iraqi government...,CNN,,narration,0.000,0.000,0.000,neutral,neutral,none,0.0,50.0,
2,He also stated that the government made extens...,,,narration,0.000,0.333,0.660,positive,positive,none,56.2,66.5,sarcasm:sure
3,"Kittleson, who specializes in Middle East repo...",,"Middle East, Kataib Hezbollah",narration,0.067,0.222,0.000,neutral,neutral,none,11.3,51.0,
4,“The U.S. Department of State extends its appr...,Rubio,"Federal Bureau, Iraqi Supreme Judicial",quote_or_quoted_content,0.000,0.000,0.700,positive,positive,none,49.0,67.5,
5,“We are relieved that this American is now fre...,Rubio,,quote_or_quoted_content,0.300,0.433,0.888,positive,positive,none,75.2,76.7,positive:support
6,"On Tuesday, Kataib Hezbollah security chief Ab...",Assaf,"On Tuesday, Kataib Hezbollah",quote_or_quoted_content,0.000,0.000,0.296,positive,positive,none,20.7,57.4,
7,According to a source familiar with the matter...,Kittleson,Kataib Hezbollah,narration,0.188,0.400,-0.772,negative,negative,none,66.0,72.1,
8,The warning came while she was already reporti...,,,narration,0.000,0.000,-0.340,negative,negative,none,23.8,58.5,
9,The abduction sparked an operation from Iraqi ...,,,narration,0.148,0.480,0.318,positive,positive,none,36.7,60.2,


CURRENT NEWS ARTICLE 2

In [ ]:
# Fill ONE of these only. Leave the other empty.

article_text = """ President Donald Trump's profanity-laced threats to wipe out Iran's civilization have led some Democrats to discuss attempting to remove him from office by using the 25th Amendment to the U.S. Constitution.
Such an effort would be an uphill struggle, since doing so would require the support of Trump's fellow Republicans, who control ​both chambers of Congress. Despite his falling overall public approval, some 82% of Republicans are happy with his presidency.
Trying to remove him from office ‌could also hold political peril for Democrats - who twice tried and failed to remove Trump from office by impeaching him during his first term.
The 25th Amendment was ratified in 1967. It was introduced after the assassination of President John F. Kennedy in 1963 and is intended to clarify the process of presidential ​succession, ensuring that the United States always has a functioning president and vice president.
The Constitution's original presidential succession clause did not address vice presidential vacancies. ​Between President George Washington's first term in 1789 and 1967, the vice presidency was vacant for more than 37 years cumulatively ⁠because of death, resignation, or succession to the presidency, according to the Congressional Research Service, opens new tab.
Presidents have invoked Section 3 of the amendment - ​dealing with circumstances in which the president is unable to discharge their responsibilities - when they knew they would be incapacitated due to medical procedures, such as in 2021, when ​then-President Joe Biden underwent a colonoscopy.
But Section 4, covering the involuntary removal of a president, has never been invoked. Section 4 allows the vice president and a majority of the president's cabinet, or, alternatively, the vice president and a majority of another unspecified body designated by Congress, to declare a president unable to discharge the powers and duties of their office.
However, if the president contests ​that decision, Congress must assemble to decide the issue within 48 hours and two-thirds majorities of both the Senate and House of Representatives must agree that the president ​is incapable. If not, the president resumes their duties.

"""

article_url = ""

# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = flagged.sort_values(["severity_score", "analysis_confidence_%", "emotion_intensity_%"], ascending=False)
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values("vader_compound").head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()

print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

AI JOURNALISM ANALYSIS REPORT
Source Type: pasted_text
Likely Outlet / Platform: Pasted Article Text
Detected Title: Not detected
Main Subject(s): President Donald Trump, President John, The Constitution, Between President George, Congressional Research Service
Total Sentences: 14
Overall Interpretation: neutral tone with noticeable bias/framing signals
Average Emotion Intensity: 28.7%
Average Analysis Confidence: 59.8%

QUICK SUMMARY
--------------------------------------------------------------------------------
Source: Pasted Article Text (pasted_text). Main subject focus: President Donald Trump, President John, The Constitution. The article appears neutral tone with noticeable bias/framing signals, with 5 bias-flagged sentences, 3 context-adjusted negative sentences, and average emotion intensity of 28.7%.

SENTIMENT COUNTS
--------------------------------------------------------------------------------
sentiment_label
negative    6
neutral     4
positive    4
Name: count, dtype: i

,sentence,speaker,subject_focus,quote_type,textblob_polarity,textblob_subjectivity,vader_compound,sentiment_label,context_adjusted_label,bias_label,emotion_intensity_%,analysis_confidence_%,matched_cues
0,President Donald Trump's profanity-laced threa...,,President Donald Trump,narration,0.000,0.000,-0.421,negative,negative,explicit_negative,29.5,68.5,explicit:threat
1,Constitution.,,,narration,0.000,0.000,0.000,neutral,context_negative,contextual_negative,0.0,50.0,
2,"Such an effort would be an uphill struggle, si...",,,narration,0.000,0.500,0.103,positive,positive,none,22.2,52.6,positive:support
3,"Despite his falling overall public approval, s...",,,narration,0.267,0.356,0.804,positive,positive,none,67.0,74.1,
4,Trying to remove him from office ‌could also h...,,,narration,-0.083,0.244,-0.718,negative,negative,implicit_negative,57.6,74.2,implicit:failed to
5,The 25th Amendment was ratified in 1967.,,,narration,0.000,0.000,0.153,positive,positive,none,10.7,53.8,
6,It was introduced after the assassination of P...,,President John,narration,0.000,0.000,-0.599,negative,negative,none,42.0,65.0,
7,Kennedy in 1963 and is intended to clarify the...,,,narration,0.000,0.000,0.599,positive,positive,none,42.0,65.0,
8,The Constitution's original presidential succe...,,"The Constitution, Between President George",narration,0.315,0.509,-0.296,negative,negative,none,37.4,62.1,
9,Presidents have invoked Section 3 of the amend...,,President Joe Biden,narration,-0.156,0.344,-0.440,negative,negative,none,41.1,63.4,


SCIENCE ARTICLE 1

In [ ]:
# Fill ONE of these only. Leave the other empty.

article_text = """
"""

article_url = "https://www.sciencedaily.com/releases/2026/04/260408225934.htm"

# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = df[df["bias_label"] != "none"].copy()
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values("vader_compound").head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()

print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

AI JOURNALISM ANALYSIS REPORT
Source Type: website
Likely Outlet / Platform: sciencedaily.com
Detected Title: Scientists just found a hidden âdrainâ inside the human brain | ScienceDaily
Main Subject(s): Medical University, South Carolina, Onder Albayram, Laboratory Medicine, Body
The
Total Sentences: 51
Overall Interpretation: mostly positive
Average Emotion Intensity: 30.1%
Average Analysis Confidence: 58.0%

QUICK SUMMARY
--------------------------------------------------------------------------------
Source: sciencedaily.com (website). Headline/title detected: Scientists just found a hidden âdrainâ inside the human brain | ScienceDaily. Main subject focus: Medical University, South Carolina, Onder Albayram. The article appears mostly positive, with 4 bias-flagged sentences, 4 context-adjusted negative sentences, and average emotion intensity of 30.1%.

SENTIMENT COUNTS
--------------------------------------------------------------------------------
sentiment_label
positive 

,sentence,speaker,subject_focus,quote_type,textblob_polarity,textblob_subjectivity,vader_compound,sentiment_label,context_adjusted_label,bias_label,emotion_intensity_%,analysis_confidence_%,matched_cues
0,How does the brain get rid of waste?,,,narration,-0.200,0.000,-0.421,negative,negative,none,29.5,63.5,
1,It relies on a specialized drainage network kn...,,,narration,0.000,0.000,0.000,neutral,context_negative,contextual_negative,0.0,50.0,
2,Scientists have been working to understand how...,,,narration,0.136,0.455,0.000,positive,positive,none,23.2,52.0,
3,A new study published in iScience by researche...,,"Medical University, South Carolina",narration,0.077,0.358,0.000,neutral,neutral,none,16.1,51.2,
4,The structure is the middle meningeal artery (...,,,narration,-0.067,0.333,-0.026,neutral,neutral,none,14.7,51.6,
5,"The research team, led by Onder Albayram, Ph.D...",,Onder Albayram,narration,0.400,0.500,0.250,positive,positive,none,43.0,62.2,
6,These imaging techniques were originally desig...,,,narration,0.188,0.425,0.000,positive,positive,none,25.9,52.8,
7,"Using this technology, the team monitored the ...",,,narration,0.500,0.500,0.402,positive,positive,none,50.0,67.5,
8,What they observed was unexpected.,,,narration,0.100,1.000,0.000,positive,positive,none,37.0,51.5,
9,"The fluid moved slowly and steadily, unlike bl...",,,narration,0.050,0.375,0.361,positive,positive,none,36.5,59.8,


SCIENCE ARTICLE 2

In [ ]:
# Fill ONE of these only. Leave the other empty.

article_text = """The search for life on Mars involves the efforts of scientists from many different disciplines. An important aspect of that search is to study Martian sedimentary rocks for information about the planet’s environment when it is likely that the surface environment hosted abundant water and therefore more habitable, around three to four billion years ago. Now, new research published in the journal Geology shows evidence of an intense sandstorm that swept through Mars’ Gale crater over three billion years ago.

“Everybody knows that the wind blew on Mars. There was an atmosphere, so it must have moved, forming breezes and gusts, and so there must have been storms, too. But this is the first definitive evidence that we’ve found of such a sandstorm,” says Steven Banham, a planetary geologist at Imperial College London and lead author of the new study. “While it does not contribute to proving existence of life on Mars, it helps paint a rich picture of the ancient surface environment.”
The finding comes from the discovery of ripple structures by Banham and a team of scientists working with the Curiosity rover. These windblown sedimentary structures were formed in a desert environment, and resemble millimeter-thick “crinkly” laminations, says Banham. Wind ripple strata like this are rarely found on Earth and have never before been observed on Mars. They can only be formed when sustained winds move large amounts of loose sand. While most sedimentary structures preserved in desert rocks on Earth or Mars record longer-term trends from seasonal winds to several thousands of years, supercritical climbing wind ripples document evidence of storms that lasted only minutes to hours.

“The thing that absolutely amazes me, is you just think that on a Tuesday afternoon, sometime, maybe 3.6 billion or so years ago, there was a sandstorm that rolled into Gale crater,” says Banham. “It would be like one of those scenes in [the movie] Dune where there’s a sandstorm happening and these ripple structures would be forming as a result. Then maybe the next day, the wind returns to normal, and it’s just another sunny day in Gale crater. But that sandstorm happened, and we have the physical evidence for it here.”

The discovery involved a degree of luck. As the Curiosity rover navigates the Martian surface, a rotating team of scientists monitor its camera and other instruments. Banham and his colleagues, including Linda Kah from the University of Tennessee and Joel Davis from Imperial College London, noticed unusual features in a black and white panorama taken at the end of each drive. The team decided to target them with higher-resolution MASTCAM cameras. Upon closer inspection, they realized they were looking at unique ripples.

“This was very serendipitous. We weren’t really looking for these deposits, and then lo and behold, we drove around the corner and found them,” says Banham. “We were lucky that we had just the right people on shift that recognized them.”
"""

article_url = ""

# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = df[df["bias_label"] != "none"].copy()
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values("vader_compound").head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()
display(HTML("<h3>Framing Trend (3-sentence rolling average)</h3>"))
print("More negative values suggest stronger negative framing across nearby sentences.")
display(df[["sentence", "framing_score", "rolling_framing", "severity_score"]].head(12))
print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

AI JOURNALISM ANALYSIS REPORT
Source Type: pasted_text
Likely Outlet / Platform: Pasted Article Text
Detected Title: Not detected
Main Subject(s): Imperial College London, Steven Banham, Linda Kah, Joel Davis
Total Sentences: 24
Overall Interpretation: mostly positive
Average Emotion Intensity: 28.5%
Average Analysis Confidence: 56.1%

QUICK SUMMARY
--------------------------------------------------------------------------------
Source: Pasted Article Text (pasted_text). Main subject focus: Imperial College London, Steven Banham, Linda Kah. The article appears mostly positive, with 0 bias-flagged sentences, 0 context-adjusted negative sentences, and average emotion intensity of 28.5%.

SENTIMENT COUNTS
--------------------------------------------------------------------------------
sentiment_label
positive    15
neutral      7
negative     2
Name: count, dtype: int64

BIAS COUNTS
--------------------------------------------------------------------------------
bias_label
none    24
Name

,sentence,speaker,subject_focus,quote_type,textblob_polarity,textblob_subjectivity,vader_compound,sentiment_label,context_adjusted_label,bias_label,emotion_intensity_%,analysis_confidence_%,matched_cues
0,The search for life on Mars involves the effor...,,,narration,0.250,0.550,0.000,positive,positive,none,34.0,53.8,
1,An important aspect of that search is to study...,,,quote_or_quoted_content,0.375,0.863,0.202,positive,positive,none,52.1,60.7,
2,"Now, new research published in the journal Geo...",,,quote_or_quoted_content,0.168,0.727,0.077,positive,positive,none,33.6,54.5,
3,“Everybody knows that the wind blew on Mars.,,,quote_or_quoted_content,0.000,0.000,0.000,neutral,neutral,none,0.0,50.0,
4,"There was an atmosphere, so it must have moved...",,,narration,0.000,0.000,0.000,neutral,neutral,none,0.0,50.0,
5,But this is the first definitive evidence that...,,"Steven Banham, Imperial College London",quote_or_quoted_content,0.129,0.429,0.000,positive,positive,none,21.9,51.9,
6,“While it does not contribute to proving exist...,,,quote_or_quoted_content,0.375,0.750,0.735,positive,positive,none,74.0,74.0,
7,The finding comes from the discovery of ripple...,,,narration,0.000,0.000,0.000,neutral,neutral,none,0.0,50.0,
8,These windblown sedimentary structures were fo...,,,quote_or_quoted_content,0.000,0.000,0.000,neutral,neutral,none,0.0,50.0,
9,Wind ripple strata like this are rarely found ...,,,narration,0.300,0.900,0.361,positive,positive,none,52.3,63.5,


FINANCE ARTICLE 1

In [ ]:
# Fill ONE of these only. Leave the other empty.

article_text = """
"""

article_url = "https://www.cnbc.com/2026/04/09/oil-prices-today-wti-brent-iran-accuse-us-of-ceasefire-breach.html"

# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = df[df["bias_label"] != "none"].copy()
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values("vader_compound").head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()

print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

AI JOURNALISM ANALYSIS REPORT
Source Type: website
Likely Outlet / Platform: cnbc.com
Detected Title: Oil rally loses steam after Israel agrees to negotiate with Lebanon
Main Subject(s): West Texas Intermediate, Abu Dhabi National, Oil Company, Sultan Ahmed Al, Prime Minister Benjamin
Total Sentences: 28
Overall Interpretation: neutral tone with noticeable bias/framing signals
Average Emotion Intensity: 25.0%
Average Analysis Confidence: 58.0%

QUICK SUMMARY
--------------------------------------------------------------------------------
Source: cnbc.com (website). Headline/title detected: Oil rally loses steam after Israel agrees to negotiate with Lebanon. Main subject focus: West Texas Intermediate, Abu Dhabi National, Oil Company. The article appears neutral tone with noticeable bias/framing signals, with 8 bias-flagged sentences, 6 context-adjusted negative sentences, and average emotion intensity of 25.0%.

SENTIMENT COUNTS
---------------------------------------------------------

,sentence,speaker,subject_focus,quote_type,textblob_polarity,textblob_subjectivity,vader_compound,sentiment_label,context_adjusted_label,bias_label,emotion_intensity_%,analysis_confidence_%,matched_cues
0,U.S. oil prices rose Thursday but pulled back ...,,,narration,0.300,0.475,0.392,positive,positive,none,41.7,64.3,
1,West Texas Intermediate crude futures for May ...,,West Texas Intermediate,narration,-0.100,0.750,-0.273,negative,negative,none,41.6,58.3,
2,International benchmark Brent crude futures fo...,,,narration,0.100,0.613,-0.572,negative,negative,none,58.4,65.8,
3,U.S. crude oil surged above $100 per barrel ea...,,,narration,-0.233,0.533,-0.745,negative,negative,none,68.2,72.1,
4,"The strait has not opened to ships, said the C...",,"Abu Dhabi National, Oil Company",narration,0.000,0.000,0.000,neutral,context_negative,contextual_negative,0.0,50.0,
5,Iran has made clear that vessels must obtain i...,Ahmed Al Jaber,Sultan Ahmed Al,narration,0.067,0.225,0.382,positive,positive,none,33.5,60.5,
6,"""That is not freedom of navigation.",,,quote_or_quoted_content,0.000,0.000,-0.522,negative,negative,none,36.5,63.0,
7,"That is coercion,"" the ADNOC chief said.",,,quote_or_quoted_content,0.000,0.000,0.000,neutral,context_negative,contextual_negative,0.0,50.0,
8,The oil rally subsequently eased after Prime M...,Minister Benjamin Netanyahu,Prime Minister Benjamin,quote_or_quoted_content,0.000,0.525,0.296,positive,positive,none,36.5,57.4,
9,Israel's military campaign in Lebanon against ...,,,narration,-0.100,0.100,-0.382,negative,negative,explicit_negative,29.7,69.0,explicit:threat


TECHNOLOGY ARTICLE 1

In [ ]:
# Fill ONE of these only. Leave the other empty.

article_text = """
"""

article_url = "https://techstartups.com/2026/04/09/top-tech-news-today-april-9-2026/"

# ==========
# RUN TOOL
# ==========

article, meta = get_article_text_and_meta(article_text, article_url)

if not article:
    raise ValueError("Please paste article_text or provide article_url.")

sentences = split_sentences_clean(article)
if len(sentences) == 0:
    raise ValueError("No usable sentences found. Try cleaner pasted text or a different URL.")

rows = []
for i, sentence in enumerate(sentences):
    prev_sentence = sentences[i - 1] if i > 0 else ""
    next_sentence = sentences[i + 1] if i + 1 < len(sentences) else ""
    rows.append(analyze_sentence(sentence, prev_sentence, next_sentence))

df = pd.DataFrame(rows)
top_subjects = extract_subjects(article, 5)
summary = summarize_article(
    df,
    source_name=meta["source_name"],
    source_type=meta["source_type"],
    title=meta["title"],
    top_subjects=top_subjects
)

print("=" * 80)
print("AI JOURNALISM ANALYSIS REPORT")
print("=" * 80)
print(f"Source Type: {summary['source_type']}")
print(f"Likely Outlet / Platform: {summary['source_name']}")
print(f"Detected Title: {summary['title'] if summary['title'] else 'Not detected'}")
print(f"Main Subject(s): {summary['top_subjects'] if summary['top_subjects'] else 'Not clearly detected'}")
print(f"Total Sentences: {summary['total_sentences']}")
print(f"Overall Interpretation: {summary['overall_article_interpretation']}")
print(f"Average Emotion Intensity: {summary['average_emotion_intensity_%']}%")
print(f"Average Analysis Confidence: {summary['average_analysis_confidence_%']}%")
print("=" * 80)

print("\nQUICK SUMMARY")
print("-" * 80)
print(summary["quick_summary"])

print("\nSENTIMENT COUNTS")
print("-" * 80)
print(df["sentiment_label"].value_counts())

print("\nBIAS COUNTS")
print("-" * 80)
print(df["bias_label"].value_counts())

print("\nQUOTE VS NARRATION")
print("-" * 80)
print(df["quote_type"].value_counts())

speaker_df = df[(df["speaker"] != "") | (df["quote_type"] == "quote_or_quoted_content")].copy()

print("\nLIKELY SPEAKERS / WHO SAID WHAT")
print("-" * 80)
if len(speaker_df) == 0:
    print("No clear speaker patterns detected.")
else:
    for _, row in speaker_df.head(8).iterrows():
        speaker_name = row["speaker"] if row["speaker"] else "Unclear / quoted speaker"
        print(f"Speaker / Subject: {speaker_name}")
        print(f"What was said: {row['sentence']}")
        print(f"AI sentiment: {row['sentiment_label']} | AI bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print("-" * 80)

print("\nTOP BIAS-FLAGGED SENTENCES")
print("-" * 80)
flagged = df[df["bias_label"] != "none"].copy()
if len(flagged) == 0:
    print("No bias-flagged sentences found with current rules.")
else:
    flagged = flagged.sort_values(["analysis_confidence_%", "emotion_intensity_%"], ascending=False)
    for _, row in flagged.head(10).iterrows():
        print(f"[{row['bias_label']}] {row['sentence']}")
        print(f"Context-adjusted label: {row['context_adjusted_label']}")
        print(f"Emotion intensity: {row['emotion_intensity_%']}% | Confidence: {row['analysis_confidence_%']}%")
        if row["matched_cues"]:
            print(f"Matched cues: {row['matched_cues']}")
        print()

print("\nTOP NEGATIVE SENTENCES")
print("-" * 80)
for _, row in df.sort_values("vader_compound").head(5).iterrows():
    print(f"({row['vader_compound']}) {row['sentence']}")
    print(f"Bias/context: {row['bias_label']} / {row['context_adjusted_label']}")
    print()

print("\nFULL RESULTS TABLE")
print("-" * 80)
display(df)

AI JOURNALISM ANALYSIS REPORT
Source Type: website
Likely Outlet / Platform: techstartups.com
Detected Title: Top Tech News Today, April 9, 2026 - Tech Startups
Main Subject(s): Why It Matters, Muse Spark, Big Tech, Alexandr Wang, The Information
Total Sentences: 135
Overall Interpretation: mostly positive
Average Emotion Intensity: 29.8%
Average Analysis Confidence: 57.6%

QUICK SUMMARY
--------------------------------------------------------------------------------
Source: techstartups.com (website). Headline/title detected: Top Tech News Today, April 9, 2026 - Tech Startups. Main subject focus: Why It Matters, Muse Spark, Big Tech. The article appears mostly positive, with 6 bias-flagged sentences, 5 context-adjusted negative sentences, and average emotion intensity of 29.8%.

SENTIMENT COUNTS
--------------------------------------------------------------------------------
sentiment_label
positive    75
neutral     36
negative    24
Name: count, dtype: int64

BIAS COUNTS
-----------

,sentence,speaker,subject_focus,quote_type,textblob_polarity,textblob_subjectivity,vader_compound,sentiment_label,context_adjusted_label,bias_label,emotion_intensity_%,analysis_confidence_%,matched_cues
0,"It’s Thursday, April 9, 2026, and here are the...",,,quote_or_quoted_content,0.500,0.500,0.202,positive,positive,none,50.0,62.6,
1,AI is no longer a future bet—it’s turning into...,,,quote_or_quoted_content,0.120,0.205,-0.542,negative,negative,none,44.1,65.4,
2,"In just the past 24 hours, we’ve seen Amazon q...",,,quote_or_quoted_content,0.010,0.217,0.542,positive,positive,none,44.5,63.7,
3,"At the same time, Anthropic is drawing a line ...",,,narration,-0.033,0.696,-0.077,neutral,neutral,none,26.3,52.4,
4,But it’s not just about AI labs and Big Tech.,,Big Tech,quote_or_quoted_content,0.000,0.100,0.000,neutral,neutral,none,3.0,50.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,"As the AI stack matures, infrastructure compan...",,,narration,0.000,0.000,0.077,neutral,neutral,none,5.4,51.9,
131,"If more of these combinations emerge, the next...",,,narration,0.239,0.344,0.000,positive,positive,none,27.1,53.6,
132,Why It Matters: AI consolidation is becoming i...,,Why It Matters,narration,0.450,0.850,0.026,positive,positive,none,57.0,57.4,
133,That’s your quick tech briefing for today.,,,quote_or_quoted_content,0.333,0.500,0.000,positive,positive,none,38.3,55.0,
